In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import display, Markdown

In [ ]:
load_dotenv(override=True)   

In [ ]:
openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")  
google_api_key = os.getenv("GOOGLE_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")  

if openai_api_key:
  print("OpenAI API key loaded successfully.")
else:
  print("OpenAI API key not found. Please set the OPENAI_API_KEY environment variable.")
    
if anthropic_api_key:
  print("Anthropic API key loaded successfully.") 
else:
  print("Anthropic API key not found. Please set the ANTHROPIC_API_KEY environment variable.")

if google_api_key:
  print("Google API key loaded successfully.")
else:    
  print("Google API key not found. Please set the GOOGLE_API_KEY environment variable.")

if deepseek_api_key:
  print("Deepseek API key loaded successfully.")
else:
  print("Deepseek API key not found. Please set the DEEPSEEK_API_KEY environment variable. It's optional.")

if groq_api_key:
  print("Groq API key loaded successfully.")
else:
  print("Groq API key not found. Please set the GROQ_API_KEY environment variable. It's optional.")
  


In [ ]:
request = "Come up with a question on how to open a one new business in UK related to Agentic AI Services that I can ask a number of LLMs to evaluate and provide their ideas on."
request += "Answer only the question and do not provide any additional information, no explanation."
messages = [{"role": "user", "content": request}]


In [ ]:
messages

In [ ]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
)

question = response.choices[0].message.content
display(Markdown(f"**OpenAI GPT-5 Mini Question:** {question}"))

In [ ]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [ ]:
model_name = "gpt-5-nano"

response = openai.chat.completions.create(
    model=model_name,
    messages=messages,
)
answer = response.choices[0].message.content
display(Markdown(f"**{model_name} Answer:** {answer}"))
competitors.append(model_name)
answers.append(answer)


In [ ]:
model_name = "claude-sonnet-4-5"

anthropic = Anthropic()
response = anthropic.messages.create(
    model=model_name,
    messages=messages,
    max_tokens=1000,
)
answer = response.content[0].text
display(Markdown(f"**{model_name} Answer:** {answer}"))
competitors.append(model_name)
answers.append(answer)

In [ ]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

response = gemini.chat.completions.create(
  model=model_name,
  messages=messages,
)
answer = response.choices[0].message.content

display(Markdown(f"**{model_name} Answer:** {answer}"))
competitors.append(model_name)
answers.append(answer)

In [ ]:
print("Competitors:", competitors)
print("Answers:", answers)

In [ ]:
for competitors, answers in zip(competitors, answers):
    print(f"Competitor: {competitors}\nAnswer: {answers}\n{'-'*50}")

In [ ]:
together = ""
for i, answer in enumerate(answers):
    together += f"Response from Competitor {i+1}\n"
    together += answer + "\n\n"

In [ ]:
print(together)

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""

In [ ]:
print(judge)

In [ ]:
judge_message = [{"role": "user", "content": judge}]


In [ ]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=judge_message,
)
results = response.choices[0].message.content
print(results)

In [ ]:
results_dict = json.loads(results)
ranks =  results_dict["results"]

for i, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {i+1}: {competitor}")